# Multimodal Sentiment Analysis — GPU training

Connect a free GPU runtime first (Runtime → Change runtime type → T4 GPU if offered). Free GPU availability and session duration are not guaranteed. This workflow downloads MELD from its official Hugging Face dataset mirror into the temporary Colab runtime; the dataset and recordings are not added to the source archive or GitHub. MELD is scripted television dialogue and does not itself validate sarcasm or accent robustness.

This is a separate notebook so your existing `Sentiment.ipynb` stays untouched.

In [ ]:
from google.colab import files
import os, zipfile
print('Upload multimodal-sentiment-source.zip from the project output folder.')
uploaded = files.upload()
archive = next(iter(uploaded))
with zipfile.ZipFile(archive) as z: z.extractall('/content/msa-project')
os.chdir('/content/msa-project/multimodal-sentiment-analysis')
print('Project extracted:', os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU if offered.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
import pathlib,subprocess
base=pathlib.Path('/content/meld'); ann=base/'annotations'; ann.mkdir(parents=True,exist_ok=True)
for split in ['train','dev','test']:
 url=f'https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/{split}_sent_emo.csv'
 subprocess.run(['wget','-q','-O',str(ann/f'{split}_sent_emo.csv'),url],check=True)
archive=base/'MELD.Raw.tar.gz'
subprocess.run(['wget','-c','--show-progress','-O',str(archive),'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz'],check=True)
subprocess.run(['tar','-xzf',str(archive),'-C',str(base)],check=True)
print('MELD extracted to',base)

In [ ]:
import subprocess,os
os.environ['PYTHONPATH']=os.path.abspath('src')
subprocess.run(['python','scripts/prepare_meld.py','--meld-root','/content/meld/MELD.Raw','--annotations','/content/meld/annotations','--output','/content/meld/processed'],check=True)
subprocess.run(['python','prepare_manifest.py','--input','/content/meld/processed/manifest.csv','--output','/content/meld/manifest.csv'],check=True)

In [ ]:
import subprocess,os
os.environ['PYTHONPATH']=os.path.abspath('src')
subprocess.run(['python','-m','mmsa.train','--manifest','/content/meld/manifest.csv','--output-dir','artifacts/final','--config','configs/default.json'],check=True)

## Resume after a runtime reset

Colab storage is temporary. For a long run, connect Drive manually and copy `artifacts/final/latest.pt` there after each epoch. Restore it to the same path after reconnecting, then add `--resume` to the training command. Free Colab sessions can disconnect; keep a copy of each completed checkpoint.

In [ ]:
import subprocess,os
os.environ['PYTHONPATH']=os.path.abspath('src')
subprocess.run(['python','-m','mmsa.evaluate','--manifest','/content/meld/manifest.csv','--model-dir','artifacts/final','--split','test'],check=True)

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/multimodal-sentiment-model','zip','artifacts/final')
files.download('/content/multimodal-sentiment-model.zip')